In [ ]:
from systemflow.hep.utils import hep_graph_from_spreadsheet, hep_with_updated_parameters, hep_construct_graph, dataframes_from_spreadsheet
from systemflow.hep.metrics import Productivity
from systemflow.classifier import L1TClassifier, get_passed
from systemflow.metrics import precision, recall, f1_score
from systemflow.models import density_scale_model
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import pickle as pkl

from scipy.optimize import curve_fit, approx_fprime
from multiprocess import Pool

In [ ]:
# load data from the spreadsheet which defines the structure of the workflow,
# as well as the parameters for data rates, efficiency, data reduction, and classifier performance
# ...these are taken from predictions for the Run-5 CMS
run5_det, run5_proc, run5_globals = dataframes_from_spreadsheet("configurations/cms_system_200.xlsx")
run5_smartpx_det, run5_smartpx_proc, run5_smartpx_globals = dataframes_from_spreadsheet("configurations/cms_system_200_smartpx.xlsx")

In [ ]:
run5_det

In [ ]:
# import the data predicting wall time scaling by pileup
scaling = pd.read_excel("wall time scaling.xlsx", sheet_name="Data")
# fit a polynomial to this data for CPU and GPU runtimes
fit_poly = lambda x, k3, k2, k1: k3 * x ** 3 + k2 * x ** 2 + k1 * x
k, cv = curve_fit(fit_poly, scaling["Size"], scaling["Wall Time"])
k_gpu, cv_gpu = curve_fit(fit_poly, scaling["Size"], scaling["Wall Time GPU"])

In [ ]:
# define a dictionary with functions defining the scaling of trigger runtimes with incoming data
funcs = {"Global": lambda x: fit_poly(x, *k), "Intermediate": lambda x: x / 2.0e6}
funcs_gpu = {"Global": lambda x: fit_poly(x, *k_gpu), "Intermediate": lambda x: x / 2.0e6}

In [ ]:
# Build base graph definitions using v2.0 API
ex_gpu = hep_graph_from_spreadsheet("configurations/cms_system_200.xlsx", functions=funcs_gpu)
ex_reduction = hep_graph_from_spreadsheet("configurations/cms_system_200_smartpx.xlsx", functions=funcs_gpu)
ex = hep_graph_from_spreadsheet("configurations/cms_system_200.xlsx", functions=funcs)
# Set ex to use L1T reduction 400:
ex = hep_with_updated_parameters(ex, {"Intermediate": {"reduction ratio (1)": 400}})

In [ ]:
def vary_parameters(graph_def, tracker_data, hgcal_data, reduction_ratio, l1t_skill, hlt_skill, l1t_eff, hlt_eff):
    l1t_cls = deepcopy(graph_def.get_node("Intermediate").parameters["classifier (obj)"])
    l1t_cls.skill_boost = l1t_skill
    hlt_cls = deepcopy(graph_def.get_node("Global").parameters["classifier (obj)"])
    hlt_cls.skill_boost = hlt_skill
    variant = hep_with_updated_parameters(graph_def, {
        "Inner Tracker": {"sample data (B)": tracker_data},
        "HGCAL": {"sample data (B)": hgcal_data},
        "Intermediate": {"reduction ratio (1)": reduction_ratio, "classifier (obj)": l1t_cls, "op efficiency (J/op)": l1t_eff},
        "Global": {"op efficiency (J/op)": hlt_eff, "classifier (obj)": hlt_cls}
    })
    result = variant()
    mv = result.metric_values
    power = mv["total power (W)"] / density_scale_model(2032)
    confusion = mv["pipeline contingency (2x2)"]
    prod = (f1_score(confusion) * 7500) / power
    return prod

In [ ]:
vp = lambda x: vary_parameters(ex_reduction, *x)

In [ ]:
# run5 system model - tracker l1t upgrade w/ smart pixels
# Execute the graph to get computed properties (message sizes)
ex_result = ex()

c_phase1 = [ex_result.get_node("Inner Tracker").properties["message size (B)"],
      ex_result.get_node("HGCAL").properties["message size (B)"],
      400, # reduction ratio
      0.0, # l1t skill boost
      0.0, # hlt skill boost
      ex.get_node("Intermediate").parameters["op efficiency (J/op)"],
      ex.get_node("Global").parameters["op efficiency (J/op)"],]

In [ ]:
grad_phase1 = approx_fprime(c_phase1,
                vp,
                [1e2,
                 1e2,
                 10, 
                 0.05,
                 0.05,
                 1e-4,
                 1e-1],
              )

In [ ]:
grad_phase1

In [ ]:
# run5 system model - tracker l1t upgrade w/ smart pixels

c_l1tracks = [ex_result.get_node("Inner Tracker").properties["message size (B)"],
      ex_result.get_node("HGCAL").properties["message size (B)"],
      400, # reduction ratio
      0.0, # l1t skill boost
      0.0, # hlt skill boost
      ex.get_node("Intermediate").parameters["op efficiency (J/op)"],
      ex.get_node("Global").parameters["op efficiency (J/op)"],]

In [ ]:
# final system model - tracker l1t upgrade w/ smart pixels
ex_reduction_result = ex_reduction()

c0a = [ex_reduction_result.get_node("Inner Tracker").properties["message size (B)"],
      ex_reduction_result.get_node("HGCAL").properties["message size (B)"],
      53.3, # reduction ratio
      0.4, # l1t skill boost
      0.0, # hlt skill boost
      ex_reduction.get_node("Intermediate").parameters["op efficiency (J/op)"],
      ex_reduction.get_node("Global").parameters["op efficiency (J/op)"],]

In [ ]:
c_sphase2 = approx_fprime(c0a,
                vp,
                [1e2,
                 1e2,
                 15, 
                 0.05,
                 0.05,
                 1e-4,
                 1],
              )

In [ ]:
c_sphase2 * 1e3

In [ ]:
c_sphase2[4]

In [ ]:
# HLT Skill / HLT energy
c_sphase2[4] / c_sphase2[6]

In [ ]:
# L1T Skill / HLT Skill
c_sphase2[3] / c_sphase2[4]

In [ ]:
p_phase2 = np.float64(0.0002920143304319754)

In [ ]:
# L1T Skill / HLT Skill
p_phase2 / c_sphase2[5]

In [ ]:
# L1T Power gradient
c_sphase2[-2] * 1e6

In [ ]:
# HLT Power gradient
c_sphase2[-1]

In [ ]:
relative_change = p_phase2 / c_sphase2

In [ ]:
relative_change

In [ ]:
# HLT skill
relative_change[4]

In [ ]:
# HLT power
relative_change[-1]